In [1]:
import gymnasium as gym
from stable_baselines3 import SAC
import torch

In [2]:
if torch.cuda.is_available():
    dev = "cuda:0"
elif torch.backends.mps.is_available():
    dev = "mps"
else:
    dev = "cpu"
device = torch.device(dev)
device

device(type='mps')

SAC does NOT work natively with CartPole, because:

* SAC requires continuous action spaces
* CartPole uses discrete actions (left/right)

So we use a continuous env:

In [3]:
env = gym.make("Pendulum-v1")

In [4]:
# model = SAC("MlpPolicy", env, verbose=1, device=device)
model = SAC("MlpPolicy", env, verbose=1, device="cpu")
model.learn(total_timesteps=20_000)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
----------------------------------
| rollout/           |           |
|    ep_len_mean     | 200       |
|    ep_rew_mean     | -1.38e+03 |
| time/              |           |
|    episodes        | 4         |
|    fps             | 388       |
|    time_elapsed    | 2         |
|    total_timesteps | 800       |
| train/             |           |
|    actor_loss      | 23.1      |
|    critic_loss     | 0.217     |
|    ent_coef        | 0.813     |
|    ent_coef_loss   | -0.32     |
|    learning_rate   | 0.0003    |
|    n_updates       | 699       |
----------------------------------
----------------------------------
| rollout/           |           |
|    ep_len_mean     | 200       |
|    ep_rew_mean     | -1.56e+03 |
| time/              |           |
|    episodes        | 8         |
|    fps             | 366       |
|    time_elapsed    | 4         |
|    total_timesteps | 1600    

In [5]:
num_eval_episodes = 100
episode_lengths = []

for episode in range(num_eval_episodes):
    state, _ = env.reset()
    done = False
    steps = 0

    while not done:
        action, _ = model.predict(state, deterministic=True)
        state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        steps += 1

    episode_lengths.append(steps)

env.close()

average_steps = sum(episode_lengths) / num_eval_episodes
print(f"Average steps over {num_eval_episodes} episodes: {average_steps:.2f}")

Average steps over 100 episodes: 200.00


In [6]:
env = gym.make("Pendulum-v1", render_mode="human")

state, _ = env.reset()
done = False

while not done:
    action, _ = model.predict(state, deterministic=True)
    state, reward, terminated, truncated, _ = env.step(action)
    done = terminated or truncated

env.close()

/Users/wick/reinforcement_learning/.venv/lib/python3.13/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
